# 🚜 Training YOLO11 Instance Segmentation untuk Dashcam Excavator
### Deteksi Dataran Tanah (Ground), Dasar Galian (Trench Floor), Tebing (Trench Wall), & Lengan Excavator

Notebook ini dioptimalkan khusus untuk melatih model **YOLO11 Instance Segmentation (SOTA)** menggunakan GPU Google Colab (T4 / A100 GPU).

**Dataset Roboflow Universe:** [`test-4vgpt/excavator-hp0jz`](https://universe.roboflow.com/test-4vgpt/excavator-hp0jz)
**Target Segmentasi Poligon:**
1. `ground` (Dataran tanah atas / permukaan kerja)
2. `trench_floor` (Dasar parit / dataran galian bawah pit)
3. `trench_wall` (Dinding lereng / tebing galian batubara & overburden)
4. `excavator_arm` (Anatomi boom, stick, & bucket)

Setelah training selesai, file `best.pt` dapat langsung diunduh dan dipasang pada server HERA AI-CCTV.

## 1. Verifikasi GPU Google Colab
Pastikan runtime menggunakan **T4 GPU** (*Runtime -> Change runtime type -> T4 GPU*).

In [ ]:
!nvidia-smi

## 2. Instalasi Dependensi (Ultralytics YOLO & Roboflow)

In [ ]:
!pip install -q "ultralytics>=8.3.0" roboflow

## 3. Unduh Dataset dari Roboflow Universe
Masukkan **Roboflow API Key** Anda pada variabel di bawah ini (dapat diambil dari *https://app.roboflow.com/settings/api*).

In [ ]:
import os
from roboflow import Roboflow

# Ganti dengan API Key Roboflow Anda
ROBOFLOW_API_KEY = "PASTE_YOUR_API_KEY_HERE"

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("test-4vgpt").project("excavator-hp0jz")
version = project.version(7)
dataset = version.download("yolov8")  # Format poligon segmentation kompatibel penuh dengan YOLO11

print(f"\nDataset berhasil diunduh di: {dataset.location}")

## 4. Konfigurasi Kelas Dataset

In [ ]:
import yaml

yaml_path = f"{dataset.location}/data.yaml"
with open(yaml_path, "r") as f:
    cfg = yaml.safe_load(f)

print("Daftar Kelas:", cfg.get("names"))
print("Jumlah Kelas:", cfg.get("nc"))

## 5. Training Model YOLO11 Instance Segmentation (`yolo11n-seg.pt`)
Model YOLO11 memanfaatkan arsitektur C3k2 & Spatial Attention terbaru untuk mengenali kontur batas tanah dan tebing galian secara presisi, bahkan dalam kondisi pencahayaan malam atau debu pit.

In [ ]:
from ultralytics import YOLO

# Menggunakan pre-trained YOLO11 Nano Segmentation (State-of-the-Art, ringan, super cepat ~3ms di GPU)
model = YOLO("yolo11n-seg.pt")

# Parameter augmentasi khusus medan tambang (debu, kemiringan lereng, variasi cahaya malam)
results = model.train(
    data=yaml_path,
    epochs=60,          # 60 epoch optimal untuk konvergensi segmentation mask
    imgsz=512,          # Resolusi 512 optimal untuk rasio dashcam vertikal/horizontal pit
    batch=16,           # Batch 16 stabil di GPU T4 16GB
    device=0,
    patience=15,        # Early stopping jika tidak ada peningkatan dalam 15 epoch
    save=True,
    project="excavator_dashcam_seg",
    name="yolo11_dashcam_terrain_v1",
    seed=42,
    # Augmentasi medan tambang
    degrees=10.0,       # Rotasi kemiringan lereng galian
    fliplr=0.5,         # Flip horizontal (gerakan lengan ayun kiri-kanan)
    hsv_h=0.015,        # Toleransi rona warna tanah
    hsv_s=0.5,          # Saturasi debu / lumpur basah
    hsv_v=0.4,          # Variasi pencahayaan siang vs sorot lampu malam
    mosaic=0.5,         # Mosaic augmentation untuk variasi skala
    verbose=True
)

## 6. Evaluasi Akurasi Segmentasi (Mask mAP@50 & mAP@50-95)

In [ ]:
# Validasi pada data validasi independen
metrics = model.val(data=yaml_path, split="val")

print("\n" + "="*50)
print("AKURASI MODEL SEGMENTASI TANAH:")
print(f"Mask mAP@50    : {metrics.seg.map50:.4f}")
print(f"Mask mAP@50-95 : {metrics.seg.map:.4f}")
print(f"Box  mAP@50    : {metrics.box.map50:.4f}")
print("="*50)

## 7. Visualisasi Hasil Prediksi Poligon Tanah pada Dashcam

In [ ]:
import glob
import matplotlib.pyplot as plt
from PIL import Image

test_images = glob.glob(f"{dataset.location}/valid/images/*.jpg")[:4]
if not test_images:
    test_images = glob.glob(f"{dataset.location}/test/images/*.jpg")[:4]

for img_path in test_images:
    preds = model.predict(source=img_path, conf=0.25, save=False)[0]
    plotted = preds.plot()
    
    plt.figure(figsize=(10, 6))
    plt.imshow(plotted)
    plt.axis("off")
    plt.title(f"Deteksi Dataran Tanah Dashcam: {os.path.basename(img_path)}")
    plt.show()

## 8. Unduh Model Terbaik (`best.pt`)
Jalankan cell ini untuk mengunduh bobot model terbaik (`best.pt`) ke komputer Anda untuk dipasang ke HERA.

In [ ]:
from google.colab import files

model_best_path = "excavator_dashcam_seg/yolo11_dashcam_terrain_v1/weights/best.pt"
if os.path.exists(model_best_path):
    print(f"Mengunduh model: {model_best_path}...")
    files.download(model_best_path)
else:
    print(f"File {model_best_path} belum ditemukan. Pastikan proses training selesai!")